In [1]:
!pip install -q --no-warn-conflicts google-adk google-auth==2.47.0

In [2]:
!pip install -q -U google-genai

In [4]:
import os
import datetime
import json
import smtplib
import re
from pathlib import Path
from email.message import EmailMessage
from google.colab import userdata
from google import genai
from google.adk.agents import Agent, LlmAgent, ParallelAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters
import ipywidgets as widgets
from IPython.display import display, clear_output
import asyncio

# Initialize the ADK Orchestration Services
session_service = InMemorySessionService()

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# 1. Environment Configuration
os.environ["GOOGLE_CLOUD_PROJECT"] = userdata.get('GOOGLE_CLOUD_PROJECT')
os.environ["CONGRESS_API_KEY"] = userdata.get('CONGRESS_API_KEY')

# 2. Define MCP Server Parameters for 2026 Official Sources
# Legislative: Official Congress.gov MCP
leg_mcp_params = StdioServerParameters(
    command="npx",
    args=["-y", "@gov-data/mcp-server-legislation"],
    env={"CONGRESS_API_KEY": os.environ["CONGRESS_API_KEY"]}
)

# Executive & Judicial: Official GPO GovInfo MCP (Federal Register & SCOTUS)
gov_mcp_params = StdioServerParameters(
    command="npx",
    args=["-y", "@gpo/mcp-server-govinfo"],
    env={"GPO_API_KEY": os.environ["CONGRESS_API_KEY"]} # GPO often uses the same data.gov key
)

# 3. Initialize Specialized Branch Agents
# We use unique output_keys to prevent data collision in the session state
legislative_agent = LlmAgent(
    name="Legislative_Expert",
    model="gemini-2.0-flash",
    instruction="Search for and retrieve full text of newly approved Bills and Acts from Congress.",
    tools=[McpToolset(connection_params=StdioConnectionParams(server_params=leg_mcp_params))],
    output_key="leg_raw_data"
)

executive_agent = LlmAgent(
    name="Executive_Expert",
    model="gemini-2.5-flash",
    instruction="Search the Federal Register for new Executive Orders and Presidential Proclamations.",
    tools=[McpToolset(connection_params=StdioConnectionParams(server_params=gov_mcp_params))],
    output_key="exe_raw_data"
)

judicial_agent = LlmAgent(
    name="Judicial_Expert",
    model="gemini-2.5-flash",
    instruction="Retrieve the syllabi and full opinions for the latest SCOTUS decisions.",
    tools=[McpToolset(connection_params=StdioConnectionParams(server_params=gov_mcp_params))],
    output_key="jud_raw_data"
)

# 4. Orchestrate with ParallelAgent
# This runs all three retrievals simultaneously to reduce total latency
tri_branch_retriever = ParallelAgent(
    name="Tri_Branch_Retriever",
    sub_agents=[legislative_agent, executive_agent, judicial_agent],
    description="Simultaneously fetches official records across all three branches of government."
)

In [7]:
# 1. Legislative Summarizer (Congress/Bills)
leg_summarizer = LlmAgent(
    name="Leg_Summarizer",
    model="claude-4.5-sonnet", # Using Claude for its superior legal reasoning
    instruction="""As an expert assistant in Congress laws and procedures, proceed to summarize the provided Legislative text ({leg_raw_data}) into Plain English.
    Focus on:Intended use, areas covered, estimated cost, direct impact on the average citizen, and any specific actions or deadlines the public needs to be aware o
    Target Audience: Regular non-legal, non-political people. Be factual and strictly unbiased.""",
    output_key="leg_summary"
)

# 2. Executive Summarizer (POTUS/Orders)
exe_summarizer = LlmAgent(
    name="Exe_Summarizer",
    model="claude-4.5-sonnet",
    instruction="""As an expert asistant in Executive Orders from the government, summarize the provided Executive Order or Proclamation ({exe_raw_data}).
    1. CORE DIRECTIVE: What is the main command or immediate impact of this order?
    2. RESPONSIBLE AGENCIES: Which government departments are tasked with carrying this out?
    3. MANDATES: What specific actions are these agencies required to take?
    4. TIMELINE: What are the key deadlines or effective dates mentioned?
    5. LEGAL BASIS: Briefly cite the authority used (e.g., specific law or Article of the Constitution).
    Ensure the tone is strictly clinical and factual. Do not interpret intent; summarize the written text.
    Target Audience: Non-legal, non-partisan, regular people. Be factual and strictly unbiased.""",
    output_key="exe_summary"
)

# 3. Judicial Summarizer (SCOTUS/Decisions)
jud_summarizer = LlmAgent(
    name="Jud_Summarizer",
    model="claude-4.5-sonnet",
    instruction="""You are a master analyst of SCOTUS pronunciation and decisions.Summarize the provided SCOTUS decision ({jud_raw_data}) for a non-legal audience.
    1. THE RULING: In plain terms, what did the Court decide?
    2. THE LEGAL REALITY: How does this change what is now legal, illegal, or required?
    3. CORE REASONING: What was the main reason the Court gave for this decision?
    4. PUBLIC IMPACT: How does this specifically affect the average citizen or a particular group?
    5. NEXT STEPS: Does this require Congress to act, or does it take effect immediately?
    Maintain strict factual neutrality. Summarize only the 'Opinion of the Court' and 'Syllabus'.""",
    output_key="jud_summary"
)

# 4. Orchestrate the Summarization Phase
tri_branch_summarizer = ParallelAgent(
    name="Tri_Branch_Summarizer",
    sub_agents=[leg_summarizer, exe_summarizer, jud_summarizer],
    description="Translates raw legal data from all three branches into plain, accessible English."
)

In [8]:
# Judge 1: The Omission Auditor (Gemini 2.5 Flash)
# Role: Cross-references raw data to ensure no "Bias by Omission."
judge_omission = LlmAgent(
    name="Judge_Omission",
    model="gemini-2.5-flash",
    instruction="""Compare the summary against the raw text ({leg_raw_data}/{exe_raw_data}/{jud_raw_data}).
    1. Check for missing legal requirements, deadlines, or fiscal costs.
    2. Assign a 'Neutrality Score' from 1-5 (5 = perfectly complete).
    Output Format: [Score: X] followed by your concise reasoning.""",
    output_key="omission_audit"
)

# Judge 2: The Tonal Auditor (Claude 4.5 Sonnet)
# Role: Scans for partisan language or "loaded" adjectives.
judge_tone = LlmAgent(
    name="Judge_Tone",
    model="claude-4.5-sonnet",
    instruction="""Analyze the summary for partisan slant or loaded adjectives.
    1. Identify words suggesting a 'good' or 'bad' outcome instead of clinical facts.
    2. Assign a 'Neutrality Score' from 1-5 (5 = strictly clinical).
    Output Format: [Score: X] followed by your concise reasoning.""",
    output_key="tone_audit"
)

# Judge 3: The Framing Auditor (GPT-5 High)
# Role: Checks if the narrative structure leads to a political conclusion.
judge_framing = LlmAgent(
    name="Judge_Framing",
    model="gpt-5-high",
    instruction="""Evaluate the summary for narrative framing bias.
    1. Does the presentation of facts lead the reader toward a specific political conclusion?
    2. Assign a 'Neutrality Score' from 1-5 (5 = perfectly balanced).
    Output Format: [Score: X] followed by your concise reasoning.""",
    output_key="framing_audit"
)

# Combine into the Professional Ensemble Bench
judicial_bench = ParallelAgent(
    name="Judicial_Bench",
    sub_agents=[judge_omission, judge_tone, judge_framing],
    description="A multi-model ensemble providing a weighted neutrality audit of legal summaries."
)

In [9]:
def calculate_consensus(session_state):
    # 1. Define the keys we are looking at
    audit_keys = ["omission_audit", "tone_audit", "framing_audit"]
    scores = []

    print("--- ⚖️ JUDICIAL CONSENSENS REPORT ---")

    for key in audit_keys:
        output = session_state.get(key, "")
        # Use regex to extract the score from the [Score: X] format
        match = re.search(r"\[Score:\s*(\d+)\]", output)
        if match:
            score = int(match.group(1))
            scores.append(score)
            print(f"{key.replace('_', ' ').title()}: {score}/5")
        else:
            print(f"⚠️ Warning: No valid score found in {key}")

    # 2. Calculate the Weighted Average
    if scores:
        average_score = sum(scores) / len(scores)
        print(f"\nFINAL NEUTRALITY SCORE: {average_score:.2f} / 5.0")

        # 3. Decision Logic
        if average_score >= 4.0:
            print("✅ STATUS: APPROVED FOR PUBLIC FEED")
            session_state["final_status"] = "Approved"
        elif average_score >= 3.0:
            print("🟡 STATUS: PENDING HUMAN REVIEW (Minor Bias Detected)")
            session_state["final_status"] = "Review Required"
        else:
            print("❌ STATUS: REJECTED (Significant Bias/Error Detected)")
            session_state["final_status"] = "Rejected"
    else:
        print("❌ ERROR: Unable to calculate consensus.")

# Example execution within the ADK workflow
# calculate_consensus(my_agent_session.state)

In [10]:
def iterative_hitl(session_state):
    # Initialize retry counter if not present
    if "retry_count" not in session_state:
        session_state["retry_count"] = 0

    status = session_state.get("final_status")
    max_retries = 2

    if status in ["Review Required", "Rejected"]:
        print(f"⚖️ RESOLUTION REQUIRED | Law ID: {session_state.get('law_id')}")
        print(f"Current Neutrality Score: {session_state.get('average_score')} | Retries: {session_state['retry_count']}/{max_retries}")

        # Use the summary generated by the agents
        text_area = widgets.Textarea(
            value=session_state.get("leg_summary", ""),
            description='Summary:',
            layout={'width': '90%', 'height': '200px'}
        )

        # UI Buttons
        btn_approve = widgets.Button(description="Approve", button_style='success')
        btn_retry = widgets.Button(description="Request New Summary", button_style='warning')
        btn_final = widgets.Button(description="Finalize Manual Edit", button_style='info')

        def log_resolution(action, final_text):
            log_entry = {
                "timestamp": datetime.datetime.now().isoformat(),
                "law_id": session_state.get("law_id"),
                "action": action,
                "retry_attempts": session_state["retry_count"],
                "final_summary": final_text[:100] + "..."
            }
            with open("audit_log.json", "a") as f:
                f.write(str(log_entry) + "\n")

        def on_retry(b):
            if session_state["retry_count"] < max_retries:
                session_state["retry_count"] += 1
                session_state["final_status"] = "Retrying"
                clear_output()
                print("🔄 Status updated to 'Retrying'. Run the summarizer cell again.")
            else:
                print("⚠️ Max retries reached. Please use 'Finalize Manual Edit'.")

        def on_approve(b):
            log_resolution("Approved as is", text_area.value)
            session_state["final_status"] = "Approved"
            clear_output()
            print("✅ Resolution Complete: Approved.")

        def on_final(b):
            log_resolution("Manual Override/Finalization", text_area.value)
            session_state["leg_summary"] = text_area.value
            session_state["final_status"] = "Approved (Manual)"
            clear_output()
            print("✍️ Resolution Complete: Manual final summary saved.")

        btn_retry.on_click(on_retry)
        btn_approve.on_click(on_approve)
        btn_final.on_click(on_final)

        # Show Retry only if under limit
        actions = [btn_approve]
        if session_state["retry_count"] < max_retries:
            actions.append(btn_retry)
        actions.append(btn_final)

        display(text_area, widgets.HBox(actions))
    else:
        print("🟢 Automatic Approval: No human intervention required.")

# To run: iterative_hitl(my_agent_session.state)

In [11]:
def post_intervention_handler(session_state):
    status = session_state.get("final_status")
    law_id = session_state.get("law_id", "Unknown_ID")

    if status == "Discarded":
        # Final audit log for a full rejection
        with open("audit_log.json", "a") as f:
            f.write(f"FINAL_REJECTION: {law_id} | Timestamp: {datetime.datetime.now()}\n")

        session_state.clear()
        print(f"🗑️ System cleanup complete for {law_id}. Moving to next item.")

    elif status and status.startswith("Approved"):
        # This prepares the data for the upcoming GitHub/Ghost cells
        print(f"🚀 {law_id} is verified (Status: {status}). Ready for GitHub and Ghost.")

# To run: post_intervention_handler(my_agent_session.state)

In [12]:
# A small sample queue to test the full pipeline
legal_queue = {
    "TEST-HR-123": """
    SECTION 1. SHORT TITLE.
    This Act may be cited as the 'Clean Energy Innovation Act'.
    SECTION 2. FINDINGS.
    Congress finds that transition to clean energy is essential for national security.
    SECTION 3. GRANTS.
    The Secretary shall award grants to eligible entities for the development
    of next-generation battery storage technologies.
    $500,000,000 is authorized for fiscal year 2026.
    """
}

In [13]:
#Master Loop

async def run_tri_branch_audit(queue):
    # Initialize the Runner right here to satisfy the parameter requirements
    # We provide both the app_name and a placeholder 'agent' or just the session
    adk_runner = Runner(session_service=session_service)

    print(f"🚀 Initializing Tri-Branch Intelligence System... {len(queue)} items in queue.")

    for law_id, raw_data in queue.items():
        print(f"\n--- Processing: {law_id} ---")

        # 1. State Initialization
        session_state = {"law_id": law_id, "raw_data": raw_data}

        # 2. RUN THE SUMMARIZERS
        print("✍️ Generating Specialized Summaries...")
        session_state = adk_runner.run(
            session_state,
            agent=tri_branch_summarizer,
            app_name="TriBranch_Intelligence"
        )

        # 3. RUN THE JUDICIAL BENCH
        print("⚖️ Running Judicial Bench Audit...")
        session_state = adk_runner.run(
            session_state,
            agent=judicial_bench,
            app_name="TriBranch_Intelligence"
        )

        # 4. Run Consensus Logic
        calculate_consensus(session_state)

        # 5. Trigger HITL Dashboard
        iterative_hitl(session_state)

        # 6. Handle Final Result
        post_intervention_handler(session_state)

# Fire it up with your test case
# await run_tri_branch_audit(legal_queue)